# Credit Risk Prediction System - Model Training and Comparison

This notebook guides you through the machine learning pipeline, including preprocessing, model training, evaluation, comparison, and export.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Load and Clean Dataset

In [ ]:
df = pd.read_csv('../dataset/credit_risk_dataset.csv')
print(f"Initial dataset size: {df.shape}")
df = df.drop_duplicates().reset_index(drop=True)
print(f"After removing duplicates: {df.shape}")

## 2. Define Features & Target split

In [ ]:
X = df.drop(columns=['Risk Level'])
y = df['Risk Level']

target_mapping = {'Low Risk': 0, 'Medium Risk': 1, 'High Risk': 2}
y_encoded = y.map(target_mapping)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=[object]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## 3. Preprocessing (Imputing, Outlier Treatment, Scaling & Encoding)

In [ ]:
# Imputation
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

X_train_num = pd.DataFrame(num_imputer.fit_transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
X_test_num = pd.DataFrame(num_imputer.transform(X_test[num_cols]), columns=num_cols, index=X_test.index)
X_train_cat = pd.DataFrame(cat_imputer.fit_transform(X_train[cat_cols]), columns=cat_cols, index=X_train.index)
X_test_cat = pd.DataFrame(cat_imputer.transform(X_test[cat_cols]), columns=cat_cols, index=X_test.index)

# Outlier Capping (Winsorization)
for col in num_cols:
    lower_bound = X_train_num[col].quantile(0.01)
    upper_bound = X_train_num[col].quantile(0.99)
    X_train_num[col] = np.clip(X_train_num[col], lower_bound, upper_bound)
    X_test_num[col] = np.clip(X_test_num[col], lower_bound, upper_bound)

# Encoding
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_train_cat_encoded = encoder.fit_transform(X_train_cat)
X_test_cat_encoded = encoder.transform(X_test_cat)
encoded_cat_names = encoder.get_feature_names_out(cat_cols)
X_train_cat_df = pd.DataFrame(X_train_cat_encoded, columns=encoded_cat_names, index=X_train.index)
X_test_cat_df = pd.DataFrame(X_test_cat_encoded, columns=encoded_cat_names, index=X_test.index)

# Scaling
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)
X_test_num_scaled = scaler.transform(X_test_num)
X_train_num_df = pd.DataFrame(X_train_num_scaled, columns=num_cols, index=X_train.index)
X_test_num_df = pd.DataFrame(X_test_num_scaled, columns=num_cols, index=X_test.index)

# Concatenate
X_train_processed = pd.concat([X_train_num_df, X_train_cat_df], axis=1)
X_test_processed = pd.concat([X_test_num_df, X_test_cat_df], axis=1)
print(f"Final features: {X_train_processed.shape[1]}")

## 4. Train Models & Evaluate

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1),
    'XGBoost': XGBClassifier(eval_metric='mlogloss', random_state=42, n_jobs=-1),
    'Support Vector Machine': SVC(probability=True, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_processed, y_train)
    preds = model.predict(X_test_processed)
    probs = model.predict_proba(X_test_processed)
    
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted')
    rec = recall_score(y_test, preds, average='weighted')
    f1 = f1_score(y_test, preds, average='weighted')
    auc_score = roc_auc_score(y_test, probs, multi_class='ovr', average='weighted')
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'ROC-AUC': auc_score
    })

results_df = pd.DataFrame(results).sort_values(by='F1 Score', ascending=False)
results_df

## 5. Feature Importances (using XGBoost/Random Forest)

In [ ]:
# Assuming XGBoost or Random Forest was trained, let's use the Random Forest importances
best_rf = models['Random Forest']
importances = best_rf.feature_importances_
feat_importances = pd.Series(importances, index=X_train_processed.columns)
feat_importances.nlargest(10).plot(kind='barh', color='purple')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.xlabel('Importance Value')
plt.show()